In [1]:
# === OPQN + VGGFace2 SETUP (fresh runtime) ===
import os, sys, shutil, random
from pathlib import Path

# GPU check (optional)
import torch
print("CUDA:", torch.cuda.is_available())

# Clone OPQN
if not os.path.exists('/content/opqn'):
    !git clone https://github.com/mzhang367/opqn.git /content/opqn
sys.path.append('/content/opqn')

# Patch 1: args.c -> args.cross_dataset
p = "/content/opqn/opqn_main.py"
s = open(p).read()
fixed = s.replace("args.c", "args.cross_dataset")
if s != fixed:
    open(p, "w").write(fixed)
    print("✅ Patched opqn_main.py (args.c -> args.cross_dataset)")

# Patch 2: make per-sample transforms fixed-size (avoid DataLoader stack error)
dl = "/content/opqn/data_loader.py"
code = """import torch
import torchvision.transforms as transforms
from torchvision import datasets
import os

def get_datasets_transform(dataset, data_dir="./data", cross_eval=False):
    if dataset != "vggface2":
        trainPaths = os.path.join(data_dir, dataset, "train")
        testPaths  = os.path.join(data_dir, dataset, "test")
    else:
        if cross_eval:
            trainPaths = os.path.join(data_dir, "vggface2", "cross_train")
            testPaths  = os.path.join(data_dir, "vggface2", "cross_test")
        else:
            trainPaths = os.path.join(data_dir, "vggface2", "train")
            testPaths  = os.path.join(data_dir, "vggface2", "test")

    if dataset == "vggface2":
        if cross_eval:
            train_tf = transforms.Compose([
                transforms.Resize(120), transforms.CenterCrop(112),
                transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)
            ])
            test_tf = train_tf
        else:
            train_tf = transforms.Compose([
                transforms.Resize(120), transforms.RandomCrop(112),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)
            ])
            test_tf = transforms.Compose([
                transforms.Resize(120), transforms.CenterCrop(112),
                transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)
            ])
    else:
        train_tf = transforms.Compose([
            transforms.Resize(35), transforms.RandomCrop(32),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(), transforms.Normalize([0.639,0.479,0.404],[0.216,0.183,0.171])
        ])
        test_tf = transforms.Compose([
            transforms.Resize(35), transforms.CenterCrop(32),
            transforms.ToTensor(), transforms.Normalize([0.639,0.479,0.404],[0.216,0.183,0.171])
        ])

    trainset = datasets.ImageFolder(root=trainPaths, transform=train_tf)
    testset  = datasets.ImageFolder(root=testPaths,  transform=test_tf)

    transform_train = torch.nn.Identity()
    transform_test  = torch.nn.Identity()
    return {"dataset": [trainset, testset], "transform": [transform_train, transform_test]}
"""
open(dl, "w").write(code)
print("✅ data_loader.py overwritten (fixed-size per-sample transforms)")

# Kaggle auth (upload kaggle.json)
print("📤 Upload kaggle.json when the chooser pops up")
from google.colab import files
uploaded = files.upload()
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Download VGGFace2
DATA_ROOT = Path("/content/vggface2_data")
if not DATA_ROOT.exists():
    !kaggle datasets download -d hearfool/vggface2 -p /content/vggface2_data -q
    !unzip -q /content/vggface2_data/vggface2.zip -d /content/vggface2_data || true
print("✅ VGGFace2 ready")

# Build per-identity 80/20 image split (no leakage, identities in both splits)
SRC = DATA_ROOT/"train"             # original identities from dataset zip
DST = Path("/content/opqn/data/vggface2")
TR, TE = DST/"train", DST/"test"
if DST.exists():
    shutil.rmtree(DST)
TR.mkdir(parents=True, exist_ok=True)
TE.mkdir(parents=True, exist_ok=True)

def safe_link(src, dst):
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)

random.seed(42)
ids = [d for d in SRC.iterdir() if d.is_dir()]
print("Identities:", len(ids))
train_count = test_count = 0
for d in ids:
    files = [f for f in d.iterdir() if f.is_file() and f.suffix.lower() in {".jpg",".jpeg",".png"}]
    if len(files) == 0:
        continue
    random.shuffle(files)
    n_test = max(1, int(0.2 * len(files)))
    test_files = files[:n_test]
    train_files = files[n_test:]

    (TR/d.name).mkdir(exist_ok=True)
    (TE/d.name).mkdir(exist_ok=True)
    for f in train_files:
        safe_link(str(f), str(TR/d.name/f.name))
        train_count += 1
    for f in test_files:
        safe_link(str(f), str(TE/d.name/f.name))
        test_count += 1

print(f"✅ Per-identity split complete -> train images={train_count}, test images={test_count}")
print("No leakage: images are disjoint; all identities present in both splits.")

CUDA: True
Cloning into '/content/opqn'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 50 (delta 20), reused 21 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 326.76 KiB | 18.15 MiB/s, done.
Resolving deltas: 100% (20/20), done.
✅ Patched opqn_main.py (args.c -> args.cross_dataset)
✅ data_loader.py overwritten (fixed-size per-sample transforms)
📤 Upload kaggle.json when the chooser pops up


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/hearfool/vggface2
License(s): unknown
✅ VGGFace2 ready
Identities: 480
✅ Per-identity split complete -> train images=141308, test images=35090
No leakage: images are disjoint; all identities present in both splits.


In [2]:
%cd /content/opqn
# Train
!python opqn_main.py --dataset vggface2 --save vggface2_64bit_8x256.tar --len 64 --num 8 --words 256 --bs 256 --lr 0.1
# Evaluate (you can also rely on the every-5-epoch eval in training)
!python opqn_main.py -e --dataset vggface2 --load vggface2_64bit_8x256.tar --len 64 --num 8 --words 256

/content/opqn
[Configuration] Training on dataset: vggface2
  Len_bits: 64
 Batch_size: 256
 learning rate: 0.100
 num_books: 8
 num_words: 256
HyperParams:
margin: 0.500	 miu: 0.1000
==> Building model..
number of identities:  480
number of training images:  141308
number of test images:  35090
number of training batches per epoch: 552
number of testing batches per epoch: 138
num. of codebooks:  8
num. of words per book:  256
dim. of word:  256
code length: 64-bit 	 learning rate: 0.100 	 scale length: 40 	 penalty margin: 0.50 	 balance_weight: 0.100
==> Epoch: 1
Epoch 1 |  Loss: 26.4764
Epoch Completed in 1min 14s
Saving..
==> Epoch: 2
Epoch 2 |  Loss: 25.1061
Epoch Completed in 1min 10s
Saving..
==> Epoch: 3
Epoch 3 |  Loss: 23.9240
Epoch Completed in 1min 10s
Saving..
==> Epoch: 4
Epoch 4 |  Loss: 22.4551
Epoch Completed in 1min 10s
Saving..
==> Epoch: 5
Epoch 5 |  Loss: 21.0542
Epoch Completed in 1min 10s
Code generated in 0min 36s 
[Evaluate Phase] MAP: 0.64% top_k: 0.70%
Saving

In [5]:
# Evaluate (you can also rely on the every-5-epoch eval in training)
!python opqn_main.py -e --dataset vggface2 --load vggface2_64bit_8x256.tar --len 64 --num 8 --words 256

===============evaluation on model vggface2_64bit_8x256.tar===============
number of train identities:  480
number of test identities:  480
number of training images:  141308
number of test images:  35090
number of training batches per epoch: 552
number of testing batches per epoch: 138
Query completed in 51987 ms 
[Evaluate Phase] MAP: 97.26% top_k: 96.92%


In [6]:
from google.colab import drive
drive.mount('/content/drive')
shutil.copy2('/content/opqn/checkpoint/vggface2_64bit_8x256.tar', '/content/drive/MyDrive/opqn_97percent_mAP.tar')
print("✅ Done! That's all you need.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Done! That's all you need.


In [24]:
import os
size_mb = os.path.getsize('/content/opqn/checkpoint/vggface2_64bit_8x256.tar') / (1024*1024)
print(f"Model size: {size_mb:.1f} MB")

Model size: 243.0 MB


In [15]:
#!/usr/bin/env python3
"""
OPQN (Orthogonal Product Quantization Network) Inference Script for VGGFace2
This script loads a trained OPQN model and performs face image retrieval.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as transforms
import numpy as np
from pathlib import Path
import sys
from typing import List, Tuple, Optional
import time

# Add OPQN to path
sys.path.append('/content/opqn')
from backbone import resnet20_pq

class OPQNModel:
    """OPQN Model wrapper for inference"""

    def __init__(self, checkpoint_path: str, device: str = 'cuda'):
        """
        Initialize OPQN model from checkpoint

        Args:
            checkpoint_path: Path to the .tar checkpoint file
            device: Device to run inference on ('cuda' or 'cpu')
        """
        self.device = device if torch.cuda.is_available() else 'cpu'
        self.checkpoint_path = checkpoint_path

        # Load checkpoint
        print(f"📦 Loading checkpoint from {checkpoint_path}")
        self.checkpoint = torch.load(checkpoint_path, map_location='cpu')

        # Extract configuration from checkpoint
        self.num_books = self.checkpoint['mlp'].shape[0]  # 8 codebooks
        self.num_words = self.checkpoint['mlp'].shape[1]  # 256 words per codebook
        self.subspace_dim = self.checkpoint['mlp'].shape[2]  # 256 dimensions per subspace
        self.feature_dim = 2048  # ResNet output dimension

        print(f"✅ Model Configuration:")
        print(f"  - Feature dimension: {self.feature_dim}")
        print(f"  - Number of codebooks: {self.num_books}")
        print(f"  - Words per codebook: {self.num_words}")
        print(f"  - Subspace dimension: {self.subspace_dim}")
        print(f"  - Hash code length: {self.num_books * int(np.log2(self.num_words))} bits")

        # Initialize backbone
        self.backbone = self._init_backbone()

        # Load MLP weights (projection matrices for each codebook)
        self.mlp_weights = self.checkpoint['mlp'].to(self.device)

        # Setup transforms
        self.transform = self._get_transform()

    def _init_backbone(self) -> nn.Module:
        """Initialize and load the backbone network"""
        net = resnet20_pq(num_layers=20, feature_dim=self.feature_dim)
        net = nn.DataParallel(net).to(self.device)
        net.load_state_dict(self.checkpoint['backbone'])
        net.eval()
        print("✅ Backbone loaded successfully")
        return net

    def _get_transform(self) -> transforms.Compose:
        """Get the image transformation pipeline"""
        return transforms.Compose([
            transforms.Resize(120),
            transforms.CenterCrop(112),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def extract_features(self, image_path: str) -> torch.Tensor:
        """
        Extract deep features from an image

        Args:
            image_path: Path to the image file

        Returns:
            Feature tensor of shape (feature_dim,)
        """
        img = Image.open(image_path).convert('RGB')
        img_tensor = self.transform(img).unsqueeze(0).to(self.device)

        with torch.no_grad():
            features = self.backbone(img_tensor)

        return features.squeeze(0)

    def features_to_hash(self, features: torch.Tensor) -> np.ndarray:
        """
        Convert features to binary hash codes using Product Quantization

        Args:
            features: Feature tensor of shape (batch_size, feature_dim) or (feature_dim,)

        Returns:
            Binary hash codes as numpy array
        """
        if features.dim() == 1:
            features = features.unsqueeze(0)

        batch_size = features.shape[0]
        hash_codes = []

        with torch.no_grad():
            # Split features into subspaces
            subspace_size = self.feature_dim // self.num_books

            for book_idx in range(self.num_books):
                # Extract subspace features
                start_idx = book_idx * subspace_size
                end_idx = (book_idx + 1) * subspace_size
                subspace_features = features[:, start_idx:end_idx]

                # Project using MLP weights for this codebook
                mlp_weight = self.mlp_weights[book_idx]  # Shape: (num_words, subspace_dim)

                # Compute distances to all codewords
                # We need to match dimensions: subspace_features is (batch, 256), mlp_weight is (256, 256)
                projected = F.linear(subspace_features, mlp_weight.T)  # Shape: (batch, num_words)

                # Find nearest codeword
                _, indices = torch.max(projected, dim=1)
                hash_codes.append(indices.cpu().numpy())

        # Stack hash codes
        hash_codes = np.stack(hash_codes, axis=1)  # Shape: (batch_size, num_books)

        return hash_codes.squeeze() if batch_size == 1 else hash_codes

    def compute_similarity(self, query_hash: np.ndarray, database_hashes: np.ndarray) -> np.ndarray:
        """
        Compute Hamming distance between query and database hashes

        Args:
            query_hash: Query hash code of shape (num_books,)
            database_hashes: Database hash codes of shape (num_samples, num_books)

        Returns:
            Similarity scores (inverse Hamming distance)
        """
        if query_hash.ndim == 1:
            query_hash = query_hash.reshape(1, -1)

        # Compute Hamming distance
        hamming_dist = np.sum(query_hash != database_hashes, axis=1)

        # Convert to similarity (lower distance = higher similarity)
        similarity = self.num_books - hamming_dist

        return similarity

    def retrieve_similar(self,
                        query_path: str,
                        database_paths: List[str],
                        top_k: int = 10) -> List[Tuple[str, float]]:
        """
        Retrieve top-k similar images from database

        Args:
            query_path: Path to query image
            database_paths: List of paths to database images
            top_k: Number of top results to return

        Returns:
            List of (image_path, similarity_score) tuples
        """
        print(f"\n🔍 Processing query image: {query_path}")

        # Extract query features and hash
        query_features = self.extract_features(query_path)
        query_hash = self.features_to_hash(query_features)

        # Process database
        print(f"📊 Processing {len(database_paths)} database images...")
        database_hashes = []

        for i, db_path in enumerate(database_paths):
            if (i + 1) % 100 == 0:
                print(f"  Processed {i + 1}/{len(database_paths)} images...")

            db_features = self.extract_features(db_path)
            db_hash = self.features_to_hash(db_features)
            database_hashes.append(db_hash)

        database_hashes = np.stack(database_hashes)

        # Compute similarities
        similarities = self.compute_similarity(query_hash, database_hashes)

        # Get top-k results
        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append((database_paths[idx], float(similarities[idx])))

        return results


def demo_retrieval(checkpoint_path: str, data_path: str = '/content/opqn/data/vggface2'):
    """
    Demo function to test image retrieval

    Args:
        checkpoint_path: Path to OPQN checkpoint
        data_path: Path to VGGFace2 data directory
    """
    print("=" * 50)
    print("🚀 OPQN Face Image Retrieval Demo")
    print("=" * 50)

    # Initialize model
    model = OPQNModel(checkpoint_path)

    # Get test images
    test_dir = Path(data_path) / 'test'
    test_images = []

    print(f"\n📂 Loading test images from {test_dir}")
    for identity_dir in list(test_dir.iterdir())[:50]:  # Use 50 identities for demo
        if identity_dir.is_dir():
            identity_images = [str(p) for p in identity_dir.iterdir()
                             if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
            test_images.extend(identity_images[:10])  # Max 10 images per identity

    print(f"✅ Loaded {len(test_images)} test images")

    if len(test_images) < 2:
        print("❌ Not enough test images for retrieval demo")
        return

    # Select query and database
    query_image = test_images[0]
    database_images = test_images[1:min(201, len(test_images))]  # Use up to 200 database images

    # Perform retrieval
    start_time = time.time()
    results = model.retrieve_similar(query_image, database_images, top_k=10)
    retrieval_time = time.time() - start_time

    # Display results
    print(f"\n✅ Retrieval completed in {retrieval_time:.2f} seconds")
    print(f"\n📊 Top 10 Similar Images:")
    print("-" * 50)

    query_identity = Path(query_image).parent.name
    print(f"Query Identity: {query_identity}")
    print(f"Query Image: {Path(query_image).name}")
    print("-" * 50)

    for rank, (img_path, similarity) in enumerate(results, 1):
        retrieved_identity = Path(img_path).parent.name
        match_indicator = "✅" if retrieved_identity == query_identity else "❌"
        print(f"{rank:2d}. {match_indicator} Identity: {retrieved_identity:15s} | "
              f"Image: {Path(img_path).name:20s} | Similarity: {similarity:.2f}/{model.num_books}")

    # Calculate precision@k for same identity
    same_identity_count = sum(1 for img_path, _ in results
                             if Path(img_path).parent.name == query_identity)
    precision = same_identity_count / len(results)
    print(f"\n📈 Precision@{len(results)}: {precision:.2%} (same identity matches)")


def batch_evaluation(checkpoint_path: str,
                     data_path: str = '/content/opqn/data/vggface2',
                     num_queries: int = 100):
    """
    Perform batch evaluation on test set

    Args:
        checkpoint_path: Path to OPQN checkpoint
        data_path: Path to VGGFace2 data directory
        num_queries: Number of queries to evaluate
    """
    print("=" * 50)
    print("📊 OPQN Batch Evaluation")
    print("=" * 50)

    # Initialize model
    model = OPQNModel(checkpoint_path)

    # Load test set
    test_dir = Path(data_path) / 'test'
    identity_to_images = {}

    print(f"\n📂 Loading test set from {test_dir}")
    for identity_dir in test_dir.iterdir():
        if identity_dir.is_dir():
            images = [str(p) for p in identity_dir.iterdir()
                     if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
            if len(images) >= 2:  # Need at least 2 images per identity
                identity_to_images[identity_dir.name] = images

    print(f"✅ Loaded {len(identity_to_images)} identities with sufficient images")

    # Prepare queries and database
    all_images = []
    image_to_identity = {}

    for identity, images in identity_to_images.items():
        for img_path in images:
            all_images.append(img_path)
            image_to_identity[img_path] = identity

    # Random sampling for evaluation
    np.random.seed(42)
    query_indices = np.random.choice(len(all_images),
                                   min(num_queries, len(all_images)),
                                   replace=False)

    # Evaluate
    print(f"\n🧪 Evaluating {len(query_indices)} queries...")
    precisions = []

    for i, query_idx in enumerate(query_indices):
        if (i + 1) % 10 == 0:
            print(f"  Progress: {i + 1}/{len(query_indices)}")

        query_image = all_images[query_idx]
        query_identity = image_to_identity[query_image]

        # Use all other images as database
        database_images = [img for j, img in enumerate(all_images) if j != query_idx]

        # Retrieve top-k
        results = model.retrieve_similar(query_image, database_images, top_k=10)

        # Calculate precision
        correct = sum(1 for img_path, _ in results
                     if image_to_identity[img_path] == query_identity)
        precisions.append(correct / len(results))

    # Report results
    mean_precision = np.mean(precisions)
    std_precision = np.std(precisions)

    print(f"\n📈 Evaluation Results:")
    print(f"  Mean Precision@10: {mean_precision:.2%} ± {std_precision:.2%}")
    print(f"  Queries evaluated: {len(precisions)}")


if __name__ == "__main__":
    # Path to your trained model
    checkpoint_path = '/content/opqn/checkpoint/vggface2_64bit_8x256.tar'

    # Run demo
    demo_retrieval(checkpoint_path)

    # Uncomment to run batch evaluation
    # batch_evaluation(checkpoint_path, num_queries=50)

🚀 OPQN Face Image Retrieval Demo
📦 Loading checkpoint from /content/opqn/checkpoint/vggface2_64bit_8x256.tar
✅ Model Configuration:
  - Feature dimension: 2048
  - Number of codebooks: 8
  - Words per codebook: 256
  - Subspace dimension: 256
  - Hash code length: 64 bits
✅ Backbone loaded successfully

📂 Loading test images from /content/opqn/data/vggface2/test
✅ Loaded 500 test images

🔍 Processing query image: /content/opqn/data/vggface2/test/n000405/0089_01.jpg
📊 Processing 200 database images...
  Processed 100/200 images...
  Processed 200/200 images...

✅ Retrieval completed in 1.48 seconds

📊 Top 10 Similar Images:
--------------------------------------------------
Query Identity: n000405
Query Image: 0089_01.jpg
--------------------------------------------------
 1. ✅ Identity: n000405         | Image: 0421_01.jpg          | Similarity: 7.00/8
 2. ✅ Identity: n000405         | Image: 0395_02.jpg          | Similarity: 6.00/8
 3. ✅ Identity: n000405         | Image: 0105_01.jpg

In [21]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from pathlib import Path
import random

def show_retrieval_results(model, query_path, database_paths, top_k=10):
    """Display query image and top-k retrieval results"""

    # Perform retrieval
    results = model.retrieve_similar(query_path, database_paths, top_k=top_k)

    # Setup figure
    fig, axes = plt.subplots(2, 6, figsize=(15, 6))
    fig.suptitle('OPQN Face Retrieval Results', fontsize=16, fontweight='bold')

    # Show query image
    query_img = Image.open(query_path).convert('RGB')
    axes[0, 0].imshow(query_img)
    axes[0, 0].set_title('QUERY', fontsize=12, fontweight='bold', color='blue')
    axes[0, 0].axis('off')

    # Hide unused first row cells
    for i in range(1, 6):
        axes[0, i].axis('off')

    # Show top-10 results
    query_identity = Path(query_path).parent.name

    for idx, (img_path, similarity) in enumerate(results[:10]):
        row, col = (idx // 5) + 1, (idx % 5) + 1

        # Load and display image
        img = Image.open(img_path).convert('RGB')
        axes[row-1, col].imshow(img)

        # Check if same identity
        retrieved_identity = Path(img_path).parent.name
        is_match = retrieved_identity == query_identity

        # Set title with color coding
        title_color = 'green' if is_match else 'red'
        axes[row-1, col].set_title(f'#{idx+1} Sim:{similarity:.0f}/8',
                                   fontsize=10, color=title_color)
        axes[row-1, col].axis('off')

    plt.tight_layout()
    plt.show()

    # Print summary
    precision = sum(1 for p, _ in results if Path(p).parent.name == query_identity) / len(results)
    print(f"✅ Precision@{top_k}: {precision:.1%} | Query: {Path(query_path).name} | Identity: {query_identity}")

# Initialize model if not already done
if 'model' not in globals():
    print("Initializing OPQN model...")
    model = OPQNModel('/content/opqn/checkpoint/vggface2_64bit_8x256.tar')

# Select random query and database
test_dir = Path('/content/opqn/data/vggface2/test')
all_test_images = list(test_dir.glob('*/*.jpg'))[:500]
query_image = str(random.choice(all_test_images))
database_images = [str(p) for p in random.sample(all_test_images, 200) if str(p) != query_image]

# Display results
show_retrieval_results(model, query_image, database_images)


🔍 Processing query image: /content/opqn/data/vggface2/test/n000316/0428_01.jpg
📊 Processing 200 database images...
  Processed 100/200 images...
  Processed 200/200 images...


<Figure size 1500x600 with 12 Axes>

✅ Precision@10: 100.0% | Query: 0428_01.jpg | Identity: n000316


In [22]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from pathlib import Path
import random

def show_retrieval_results(model, query_path, database_paths, top_k=10):
    """Display query image and top-k retrieval results"""

    # Perform retrieval
    results = model.retrieve_similar(query_path, database_paths, top_k=top_k)

    # Setup figure
    fig, axes = plt.subplots(2, 6, figsize=(15, 6))
    fig.suptitle('OPQN Face Retrieval Results', fontsize=16, fontweight='bold')

    # Show query image
    query_img = Image.open(query_path).convert('RGB')
    axes[0, 0].imshow(query_img)
    axes[0, 0].set_title('QUERY', fontsize=12, fontweight='bold', color='blue')
    axes[0, 0].axis('off')

    # Hide unused first row cells
    for i in range(1, 6):
        axes[0, i].axis('off')

    # Show top-10 results
    query_identity = Path(query_path).parent.name

    for idx, (img_path, similarity) in enumerate(results[:10]):
        row, col = (idx // 5) + 1, (idx % 5) + 1

        # Load and display image
        img = Image.open(img_path).convert('RGB')
        axes[row-1, col].imshow(img)

        # Check if same identity
        retrieved_identity = Path(img_path).parent.name
        is_match = retrieved_identity == query_identity

        # Set title with color coding
        title_color = 'green' if is_match else 'red'
        axes[row-1, col].set_title(f'#{idx+1} Sim:{similarity:.0f}/8',
                                   fontsize=10, color=title_color)
        axes[row-1, col].axis('off')

    plt.tight_layout()
    plt.show()

    # Print summary
    precision = sum(1 for p, _ in results if Path(p).parent.name == query_identity) / len(results)
    print(f"✅ Precision@{top_k}: {precision:.1%} | Query: {Path(query_path).name} | Identity: {query_identity}")

# Initialize model if not already done
if 'model' not in globals():
    print("Initializing OPQN model...")
    model = OPQNModel('/content/opqn/checkpoint/vggface2_64bit_8x256.tar')

# Select random query and database
test_dir = Path('/content/opqn/data/vggface2/test')
all_test_images = list(test_dir.glob('*/*.jpg'))[:500]
query_image = str(random.choice(all_test_images))
database_images = [str(p) for p in random.sample(all_test_images, 200) if str(p) != query_image]

# Display results
show_retrieval_results(model, query_image, database_images)


🔍 Processing query image: /content/opqn/data/vggface2/test/n000049/0099_01.jpg
📊 Processing 199 database images...
  Processed 100/199 images...


<Figure size 1500x600 with 12 Axes>

✅ Precision@10: 100.0% | Query: 0099_01.jpg | Identity: n000049


In [23]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
from pathlib import Path
import random

def show_retrieval_results(model, query_path, database_paths, top_k=10):
    """Display query image and top-k retrieval results"""

    # Perform retrieval
    results = model.retrieve_similar(query_path, database_paths, top_k=top_k)

    # Setup figure
    fig, axes = plt.subplots(2, 6, figsize=(15, 6))
    fig.suptitle('OPQN Face Retrieval Results', fontsize=16, fontweight='bold')

    # Show query image
    query_img = Image.open(query_path).convert('RGB')
    axes[0, 0].imshow(query_img)
    axes[0, 0].set_title('QUERY', fontsize=12, fontweight='bold', color='blue')
    axes[0, 0].axis('off')

    # Hide unused first row cells
    for i in range(1, 6):
        axes[0, i].axis('off')

    # Show top-10 results
    query_identity = Path(query_path).parent.name

    for idx, (img_path, similarity) in enumerate(results[:10]):
        row, col = (idx // 5) + 1, (idx % 5) + 1

        # Load and display image
        img = Image.open(img_path).convert('RGB')
        axes[row-1, col].imshow(img)

        # Check if same identity
        retrieved_identity = Path(img_path).parent.name
        is_match = retrieved_identity == query_identity

        # Set title with color coding
        title_color = 'green' if is_match else 'red'
        axes[row-1, col].set_title(f'#{idx+1} Sim:{similarity:.0f}/8',
                                   fontsize=10, color=title_color)
        axes[row-1, col].axis('off')

    plt.tight_layout()
    plt.show()

    # Print summary
    precision = sum(1 for p, _ in results if Path(p).parent.name == query_identity) / len(results)
    print(f"✅ Precision@{top_k}: {precision:.1%} | Query: {Path(query_path).name} | Identity: {query_identity}")

# Initialize model if not already done
if 'model' not in globals():
    print("Initializing OPQN model...")
    model = OPQNModel('/content/opqn/checkpoint/vggface2_64bit_8x256.tar')

# Select random query and database
test_dir = Path('/content/opqn/data/vggface2/test')
all_test_images = list(test_dir.glob('*/*.jpg'))[:500]
query_image = str(random.choice(all_test_images))
database_images = [str(p) for p in random.sample(all_test_images, 200) if str(p) != query_image]

# Display results
show_retrieval_results(model, query_image, database_images)


🔍 Processing query image: /content/opqn/data/vggface2/test/n000209/0167_01.jpg
📊 Processing 199 database images...
  Processed 100/199 images...


<Figure size 1500x600 with 12 Axes>

✅ Precision@10: 100.0% | Query: 0167_01.jpg | Identity: n000209


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as transforms
import numpy as np
from pathlib import Path
import sys

# Add OPQN to path
sys.path.append('/content/opqn')
from backbone import resnet20_pq

# Copy the OPQNModel class definition here (from my inference script)
# or import it if you saved it as a .py file

# Load your saved model
model = OPQNModel('/content/drive/MyDrive/opqn_97percent_mAP.tar')
print("✅ Model loaded! Ready for inference.")